# Car Semantic Search - Sentence Transformer

## 1. Cài thư viện

In [1]:
%pip install -q sentence-transformers pandas kagglehub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import và load model
Import và load model lần chạy đầu tiên, các lần chạy sau không cần import lại.

In [2]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
print("Đã load model:", MODEL_NAME)

d:\FSB-MSA\PPR501\PPR501-car-analysis\sources\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3336.77it/s]


Đã load model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 3. Tải dữ liệu xe từ Kaggle

Notebook tải dataset công khai [qucvinhdng/automobile-dataset](https://www.kaggle.com/datasets/qucvinhdng/automobile-dataset) từ Kaggle vào cache bằng KaggleHub, không cần chuẩn bị file CSV local. File dữ liệu trong dataset là `data.csv`.

In [3]:
DATASET_HANDLE = "qucvinhdng/automobile-dataset"
dataset_dir = Path(kagglehub.dataset_download(DATASET_HANDLE))
csv_path = dataset_dir / "data.csv"

if not csv_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy data.csv trong dataset Kaggle: {dataset_dir}")

raw_df = pd.read_csv(csv_path)
required_columns = {"Name", "Description"}
missing_columns = required_columns.difference(raw_df.columns)
if missing_columns:
    raise ValueError(f"Dataset thiếu các cột bắt buộc: {sorted(missing_columns)}")

cars = raw_df.rename(columns={"Name": "name", "Description": "description"}).copy()
cars["name"] = cars["name"].fillna("").astype(str).str.strip()
cars["description"] = cars["description"].fillna("").astype(str).str.strip()
cars = cars.loc[cars["name"].ne("")].reset_index(drop=True)
if cars.empty:
    raise ValueError("Dataset Kaggle không có bản ghi xe hợp lệ.")

search_columns = [column for column in cars.columns if column not in {"Link", "Car_code"}]
cars["search_text"] = cars[search_columns].fillna("").astype(str).agg(" | ".join, axis=1)
display(cars[["name", "description"]].head())

100%|██████████| 746k/746k [00:01<00:00, 611kB/s]

Extracting files...


,name,description
0,Mercedes Benz C class C200 Exclusive - 2021,"*Xe lắp ráp trong nước, màu xanh, máy xăng 2.0..."
1,Nissan Terra E 2.5 AT 2WD - 2019,"*Xe nhập khẩu, màu vàng, máy xăng 2.5 L, số tự..."
2,Mercedes Benz GLC 300 4Matic - 2021,"*Xe lắp ráp trong nước, màu đen, máy xăng 2.0 ..."
3,Hyundai SantaFe 2.4L 4WD - 2016,"*Xe lắp ráp trong nước, màu bạc, máy xăng 2.4 ..."
4,Kia Carnival Luxury 2.2D - 2022,"*Xe lắp ráp trong nước, màu xanh, máy dầu 2.2 ..."


## 4. Chuyển thông tin xe thành vector

Mô tả và thông tin của từng tin đăng được mã hóa thành một vector có số chiều cố định. Chỉ lấy 8 phần tử đầu để dễ quan sát.

In [4]:
car_texts = cars["search_text"].tolist()
car_embeddings = model.encode(car_texts, convert_to_numpy=True, normalize_embeddings=True)
print("Kích thước ma trận embedding:", car_embeddings.shape)
print("8 giá trị đầu của vector xe đầu tiên:", np.round(car_embeddings[0][:8], 4))

Kích thước ma trận embedding: (15442, 384)
8 giá trị đầu của vector xe đầu tiên: [ 0.0268  0.083   0.0127 -0.051  -0.0281  0.0535 -0.0061  0.0593]


## 5. Tìm xe theo câu truy vấn và tính Cosine Similarity

Vì các vector đã được chuẩn hóa về độ dài 1, tích vô hướng bằng Cosine Similarity. Điểm cao hơn nghĩa là gần nhau hơn về ngữ nghĩa; không phải xác suất chính xác.

In [5]:
query = "Tôi muốn tìm xe SUV 7 chỗ rộng rãi cho gia đình đi du lịch"
query_embedding = model.encode(query, convert_to_numpy=True, normalize_embeddings=True)
scores = car_embeddings @ query_embedding

result_columns = [column for column in ["name", "Price", "Location", "description"] if column in cars.columns]
results = cars[result_columns].copy()
results["similarity"] = scores
results = results.sort_values("similarity", ascending=False).head(3)

print("Câu tìm kiếm:", query)
print("Kích thước vector câu hỏi:", query_embedding.shape)
display(results.reset_index(drop=True))

Câu tìm kiếm: Tôi muốn tìm xe SUV 7 chỗ rộng rãi cho gia đình đi du lịch
Kích thước vector câu hỏi: (384,)


,name,Price,Location,description,similarity
0,Honda CRV 2.0 AT - 2007,269 Triệu,Hải Phòng,"*Xe nhập khẩu, màu bạc, máy xăng 2.0 L, số tự ...",0.564087
1,Honda CRV E - 2019,699 Triệu,Hà Nội,"*Xe nhập khẩu, màu trắng, máy xăng 1.5 L, số t...",0.562782
2,Honda CRV L - 2019,785 Triệu,Hà Nội,"*Xe nhập khẩu, màu trắng, máy xăng 1.5 L, số t...",0.560511


## 6. Thử tìm kiếm nhiều lần

Đổi nội dung `query` rồi chạy lại cell này. Không cần encode lại danh sách xe.

In [6]:
def search_cars(query, top_k=3):
    if not query.strip():
        raise ValueError("Câu tìm kiếm không được để trống.")

    query_vector = model.encode(query.strip(), convert_to_numpy=True, normalize_embeddings=True)
    similarity = car_embeddings @ query_vector
    top_k = min(max(int(top_k), 1), len(cars))
    ranking = np.argsort(similarity)[::-1][:top_k]

    results = cars.iloc[ranking][result_columns].copy()
    results["Cosine Similarity"] = np.round(similarity[ranking], 4)
    return results.reset_index(drop=True)

query = "Tôi cần xe nhỏ gọn tiết kiệm xăng để đi làm"
display(search_cars(query))

,name,Price,Location,description,Cosine Similarity
0,Mini One 1.5 AT - 2020,1 Tỷ 200 Triệu,Hà Nội,"*Xe nhập khẩu, màu xanh, máy xăng 1.5 L, số tự...",0.5613
1,Gaz Sobol - 2024,500 Triệu,Hà Nội,"*Xe lắp ráp trong nước, màu trắng, máy dầu , s...",0.5564
2,Honda CRV G - 2024,999 Triệu,Hà Nội,"*Xe lắp ráp trong nước, màu trắng, máy xăng 1....",0.5468
